In [ ]:
# Loading the Four raw sheets

import pandas as pd

orders = pd.read_excel("data/Muesli_Project_raw_data.xlsx", sheet_name="Orders", header=1)
campaign = pd.read_excel("data/Muesli_Project_raw_data.xlsx", sheet_name="Campaign Data")
process = pd.read_excel("data/Muesli_Project_raw_data.xlsx", sheet_name="Order Process Data")
intern = pd.read_excel("data/Muesli_Project_raw_data.xlsx", sheet_name="InternData Study")

In [ ]:
orders.head()     #shows the first 5 rows of the orders DataFrame

,Index,Order ID,Order Date,Ship Mode,Customer ID,Customer Name,Origin Channel,Country/Region,City,State,Postal Code,Region,Category,Sub-Category,Product ID,Sales,Quantity,Discount,Profit
0,27,CA-2019-121755,2019-01-16,Second Class,EH-13945,Eric Hoffmann,Email,United States,Los Angeles,California,90049.0,West,Special Projects Muesil,Gluten Free,TEC-AC-10003027,90.57,3,0.0,11.7741
1,45,CA-2019-118255,2019-03-11,First Class,ON-18715,Odella Nelson,Sales,United States,Eagan,Minnesota,55122.0,Central,Special Projects Muesil,Gluten Free,TEC-AC-10000171,45.98,2,0.0,19.7714
2,48,CA-2019-169194,2019-06-20,Standard Class,LH-16900,Lena Hernandez,Email,United States,Dover,Delaware,19901.0,East,Special Projects Muesil,Gluten Free,TEC-AC-10002167,45.00,3,0.0,4.9500
3,60,CA-2019-111682,2019-06-17,First Class,TB-21055,Ted Butterfield,Email,United States,Troy,New York,12180.0,East,Special Projects Muesil,Gluten Free,TEC-AC-10002167,30.00,2,0.0,3.3000
4,63,CA-2018-135545,2018-11-24,Standard Class,KM-16720,Kunst Miller,Email,United States,Los Angeles,California,90004.0,West,Special Projects Muesil,Gluten Free,TEC-AC-10004633,13.98,2,0.0,6.1512


In [ ]:
# Checking Duplicates on each raw sheets

print("Orders — full-row duplicates:", orders.duplicated().sum())      #.sum() turns it into one whole count of how many duplicate rows exist total
print("Campaign Data — full-row duplicates:", campaign.duplicated().sum())
print("Order Process Data — full-row duplicates:", process.duplicated().sum())
print("InternData Study — full-row duplicates:", intern.duplicated().sum())

Orders — full-row duplicates: 0
Campaign Data — full-row duplicates: 0
Order Process Data — full-row duplicates: 0
InternData Study — full-row duplicates: 86


In [ ]:
# Isolate all 86 duplicate rows (showing all copies)
#  Sort by Order ID so the identical rows sit right next to each other
# View the first 10 rows to inspect them

duplicates_df = intern[intern.duplicated(keep=False)]   
duplicates_df_sorted = duplicates_df.sort_values(by='Order ID')  
duplicates_df_sorted.head(10)    

,Order ID,Ready to Ship Date,Pickup Date
78,CA-2019-100244,2019-09-24,2019-09-25
79,CA-2019-100244,2019-09-24,2019-09-25
80,CA-2019-100244,2019-09-24,2019-09-25
82,CA-2019-100244,2019-09-24,2019-09-25
83,CA-2019-100244,2019-09-24,2019-09-25
59,CA-2019-102162,2019-09-16,2019-09-18
61,CA-2019-102162,2019-09-16,2019-09-18
4,CA-2019-106278,2019-09-05,2019-09-06
6,CA-2019-106278,2019-09-05,2019-09-06
10,CA-2019-106950,2019-09-06,2019-09-09


In [ ]:
# Loop through all four sheets at a time and finding the Missing values and Duplicates.
# loop through all 4 sheets one at a time
# counts missing values per column
# keep only the columns that actually have missing values (more than 0)
# how many rows the sheet has in total and how many unique Order IDs exist

print("\n--- Missing values per sheet ---")
for name, df in [("orders", orders), ("process", process),          
                  ("intern", intern), ("campaign", campaign)]:
    na_counts = df.isna().sum()                                     
    print(f"{name}:\n{na_counts[na_counts > 0]}\n")                  
 
print("--- Duplicate Order IDs per sheet ---")
for name, df in [("orders", orders), ("process", process),
                  ("intern", intern), ("campaign", campaign)]:
    total_rows = len(df)                                           
    unique_ids = df["Order ID"].nunique()                           
    full_row_dupes = df.duplicated().sum()
    print(f"{name}: {total_rows} rows, {unique_ids} unique Order IDs, "
          f"{full_row_dupes} exact full-row duplicates")


--- Missing values per sheet ---
orders:
Postal Code    11
dtype: int64

process:
Series([], dtype: int64)

intern:
Series([], dtype: int64)

campaign:
Series([], dtype: int64)

--- Duplicate Order IDs per sheet ---
orders: 9994 rows, 5009 unique Order IDs, 0 exact full-row duplicates
process: 5899 rows, 3002 unique Order IDs, 0 exact full-row duplicates
intern: 290 rows, 204 unique Order IDs, 86 exact full-row duplicates
campaign: 333 rows, 333 unique Order IDs, 0 exact full-row duplicates


In [ ]:
# Clean duplicates
# Should show 204 rows out of 290

intern_clean = intern.drop_duplicates()
print(intern_clean.shape)                        

(204, 3)


In [ ]:
# Filling the postal code for the Missing city and state [Burlington and Vermont]
# missing_mask — a True/False column, True wherever Postal Code is missing
# .sum() counts how many Trues
# To filter down the rows that are missing a postal code, and only shows their City/State columns
# official USPS ZIP code for Burlington, VT and Assign value for Burington,Vermont rows. #providing all 11 previously-missing rows a value.

missing_mask = orders["Postal Code"].isna()    
print(f"Rows with missing Postal Code before fill: {missing_mask.sum()}")  
print(orders.loc[missing_mask, ["City", "State"]].drop_duplicates())    
orders.loc[(orders["City"] == "Burlington") & (orders["State"] == "Vermont"),  
    "Postal Code"] = 5401  
print(f"Rows with missing Postal Code after fill: {orders['Postal Code'].isna().sum()}") 

Rows with missing Postal Code before fill: 11
           City    State
173  Burlington  Vermont
Rows with missing Postal Code after fill: 0


In [ ]:
# Cleaning duplicate order ID's

# loop that goes through all 4 sheets one at a time.
# in a list of pairs ,each pair has a text label (like "orders") and the actual DataFrame it refers to (orders).
# for name, df in [] — in loop process, name becomes the label (e.g. "process"), and df becomes the real DataFrame (e.g. the process table).will make same code to run once for all instead of writing it 4 times


print("--- Duplicate Order IDs per sheet ---")
for name, df in [("orders", orders), ("process", process), ("intern", intern), ("campaign", campaign)]:
    total_rows = len(df)
    unique_ids = df["Order ID"].nunique()

    #only for this sheet, skip the first column before checking: df.iloc[:, 1:] means all rows
    #but starting from column index 1 onward everything except column 0 (Row ID).
    
    if name == "process":
        full_row_dupes = df.iloc[:, 1:].duplicated().sum()  # exclude first column (Row ID)
    else:
        full_row_dupes = df.duplicated().sum()
    
    print(f"{name}: {total_rows} rows, {unique_ids} unique Order IDs, "
          f"{full_row_dupes} exact full-row duplicates")

# now actually create the cleaned version of process (not just count duplicates, but remove them)
process_clean = process.drop_duplicates(subset=process.columns[1:])
print(f"\nprocess_clean shape: {process_clean.shape}")

--- Duplicate Order IDs per sheet ---
orders: 9994 rows, 5009 unique Order IDs, 0 exact full-row duplicates
process: 5899 rows, 3002 unique Order IDs, 2896 exact full-row duplicates
intern: 290 rows, 204 unique Order IDs, 86 exact full-row duplicates
campaign: 333 rows, 333 unique Order IDs, 0 exact full-row duplicates

process_clean shape: (3003, 5)


In [ ]:
# combine the delivery-timing data into one table
# starting point: process_clean (has Order Date + Truck Scan Date, ~3000 orders)
# then bring in whatever intern/campaign data exists for those same orders
# Calculated coverage and coverage_pct as their own separate variables first, then loops through them together

combined = (process_clean
    .merge(intern_clean, on="Order ID", how="left")
    .merge(campaign_clean, on="Order ID", how="left"))

print(f"Total orders in combined table: {len(combined)}")

coverage = combined[["Ready to Ship Date", "Pickup Date", "Arrival Scan Date"]].notna().sum()
coverage_pct = (coverage / len(combined) * 100).round(1)

print("\nData coverage per stage:")
for col in coverage.index:
    print(f"  {col}: {coverage[col]} orders / {coverage_pct[col]}% of total")

    

Total orders in combined table: 3003

Data coverage per stage:
  Ready to Ship Date: 204 orders / 6.8% of total
  Pickup Date: 204 orders / 6.8% of total
  Arrival Scan Date: 333 orders / 11.1% of total


This output (6.8%, 11.1%) is coverage relative to the 3,003-order combined table — which already excludes the ~2,000 orders that have zero process/truck data at all.

In [45]:
print(timeline.shape)   # should be (5009, 7)
timeline.head(10)

(5009, 7)


,Order ID,Order Date,Ship Mode,On Truck Scan Date,Ready to Ship Date,Pickup Date,Arrival Scan Date
0,CA-2019-121755,2019-01-16,Second Class,2019-01-23,NaT,NaT,NaT
1,CA-2019-118255,2019-03-11,First Class,2019-03-13,NaT,NaT,NaT
2,CA-2019-169194,2019-06-20,Standard Class,2019-06-26,NaT,NaT,NaT
3,CA-2019-111682,2019-06-17,First Class,2019-06-19,NaT,NaT,NaT
4,CA-2018-135545,2018-11-24,Standard Class,NaT,NaT,NaT,NaT
5,CA-2020-155558,2020-10-26,Standard Class,2020-11-04,NaT,NaT,NaT
6,CA-2019-158568,2019-08-29,Standard Class,2019-09-04,NaT,NaT,NaT
7,US-2018-156867,2018-11-13,Standard Class,NaT,NaT,NaT,NaT
8,CA-2020-119004,2020-11-23,Standard Class,2020-12-02,2020-11-30,2020-12-02,NaT
9,CA-2018-129476,2018-10-15,Standard Class,NaT,NaT,NaT,NaT


In [44]:
# turn the raw dates into actual day-counts we can analyze
combined["processing_time"] = (combined["Ready to Ship Date"] - combined["Order Date"]).dt.days
combined["ship_delay"] = (combined["On Truck Scan Date"] - combined["Ready to Ship Date"]).dt.days
combined["transit_time"] = (combined["Arrival Scan Date"] - combined["On Truck Scan Date"]).dt.days
combined["total_lead_time"] = (combined["Arrival Scan Date"] - combined["Order Date"]).dt.days

combined[['processing_time', 'ship_delay', 'transit_time', 'total_lead_time']].mean().round(2)

processing_time     4.18
ship_delay          1.57
transit_time        4.60
total_lead_time    10.83
dtype: float64